# LLM Fine-Tuning on a GPU -- Beyond Summit 2026 Prep

**Goal:** Fine-tune a small LLM using LoRA on a free Colab T4 GPU in ~20 minutes.  
**What you'll learn:** The exact workflow speakers at Beyond Summit will reference --  
quantization, LoRA adapters, SFT training, GPU memory management, and generation.

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

---

### Key Concepts (mapped to Beyond Summit talks)

| Concept | What it is | Who will talk about it tomorrow |
|---------|-----------|--------------------------------|
| **Quantization (4-bit)** | Shrink model weights from 16-bit to 4-bit to fit in GPU memory | AMD (FP8 on MI355X), Eric Hartford |
| **LoRA** | Train only small adapter matrices instead of full model | Zyphra, Eric Hartford |
| **SFTTrainer** | Supervised fine-tuning trainer from HuggingFace TRL | General -- standard tool |
| **GPU Memory** | T4 has 16GB; MI300X has 192GB; MI355X has 288GB | TensorWave, AMD |
| **torch.cuda** | PyTorch GPU API (same API works on AMD via HIP) | AMD, ZLUDA, Spectral Compute |

## 1. Install packages

These are the standard fine-tuning stack in 2026.  
On AMD GPUs, you'd swap the CUDA backend for ROCm -- the Python code stays identical.

In [ ]:
!pip install -q transformers accelerate peft trl datasets bitsandbytes

## 2. Check GPU

On Colab you get an NVIDIA T4 (16GB). At Beyond Summit, the equivalent AMD card  
is MI300X (192GB) or MI355X (288GB) -- 12-18x more memory.  
This means models that need quantization on T4 can run in full precision on AMD.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
    # On AMD with ROCm, this same code works -- torch.cuda maps to HIP
    # That's the portability story you'll hear about tomorrow
else:
    raise RuntimeError("No GPU found. Go to Runtime > Change runtime type > T4 GPU")

## 3. Load a small model with 4-bit quantization

We use **TinyLlama-1.1B** -- small enough for T4, large enough to be meaningful.  

**Quantization** shrinks model weights:
- FP32 (full): 1.1B params × 4 bytes = ~4.4 GB
- FP16 (half): 1.1B params × 2 bytes = ~2.2 GB  
- INT4 (what we use): 1.1B params × 0.5 bytes = ~0.55 GB + overhead ≈ ~0.8 GB

At Beyond Summit, AMD will talk about **FP8** quantization on MI355X --  
a middle ground between FP16 and INT4 with better accuracy.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config
# This is what lets a 1.1B model fit comfortably on a 16GB T4
# On MI300X (192GB), you could load a 70B model WITHOUT quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # normalized float 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # compute in fp16
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",  # automatically places layers on GPU
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Test Verilog generation BEFORE fine-tuning

Let's see what the base model produces for Verilog prompts.  
TinyLlama was not trained on much Verilog, so expect poor results here.

In [ ]:
def generate(prompt, max_new_tokens=200):
    """Generate text from a prompt."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Verilog-oriented test prompts
test_prompts = [
    "Write a Verilog module for a 4-bit counter with synchronous reset.",
    "Write a Verilog module for a 2-to-1 multiplexer.",
    "Write a Verilog module for a D flip-flop with enable and async reset.",
]

print("=" * 60)
print("BEFORE FINE-TUNING")
print("=" * 60)
for p in test_prompts:
    print(f"\nQ: {p}")
    print(f"A: {generate(p)}")
    print("-" * 40)

## 5. Prepare training data -- Real Verilog from GitHub

We use **`shailja/Verilog_GitHub`** -- 109k real Verilog modules scraped from GitHub.  
This is the same dataset used to train VeriGen (paper: arxiv:2212.11140).

We format it as instruction-tuning data: given a module header/description,  
the model learns to complete the Verilog implementation.

On a T4, we use 200 examples to keep training under 10 minutes.  
On MI300X/MI355X, you could train on the full 109k dataset.

In [ ]:
from datasets import load_dataset, Dataset
import re

# Load real Verilog code from HuggingFace
# 109k Verilog modules from GitHub (MIT license, BigCode OpenRAIL-M)
print("Loading Verilog dataset from HuggingFace...")
raw_ds = load_dataset("shailja/Verilog_GitHub", split="train", streaming=True)

# Process: extract module name and create instruction/completion pairs
def extract_module_info(code):
    """Extract module name and signature from Verilog code."""
    # Match module declaration
    match = re.search(r'module\s+(\w+)\s*(?:#\s*\([^)]*\))?\s*\(([^;]*)\)\s*;', code, re.DOTALL)
    if match:
        name = match.group(1)
        ports = match.group(2).strip()
        return name, ports
    return None, None

# Collect and format training examples
train_data = []
skipped = 0
for i, example in enumerate(raw_ds):
    if len(train_data) >= 200:  # 200 examples for T4; increase for bigger GPU
        break

    code = example.get("text", "") or example.get("code", "") or ""
    if len(code) < 50 or len(code) > 2000:  # skip tiny or huge files
        skipped += 1
        continue

    name, ports = extract_module_info(code)
    if not name:
        skipped += 1
        continue

    # Format as instruction: "Complete this Verilog module" -> full code
    instruction = f"Write the Verilog implementation for module `{name}` with ports: {ports}"
    response = code.strip()

    train_data.append({"instruction": instruction, "response": response})

print(f"Collected {len(train_data)} Verilog examples (skipped {skipped})")

# Format as chat conversations for SFT
def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list(train_data).map(format_example)
print(f"\nTraining examples: {len(dataset)}")
print(f"\n--- Sample instruction ---")
print(dataset[0]["instruction"] if "instruction" in dataset.column_names else train_data[0]["instruction"])
print(f"\n--- Sample Verilog (first 400 chars) ---")
print(train_data[0]["response"][:400])

## 6. Configure LoRA adapters

**LoRA (Low-Rank Adaptation)** is the key technique that makes fine-tuning practical:

- Instead of updating all 1.1B parameters, we freeze the base model
- We add small trainable matrices (adapters) to specific layers
- Typically trains only 1-5% of total parameters
- Result: same quality, fraction of the GPU memory and time

**Tomorrow's context:** When Eric Hartford or Zyphra talk about fine-tuning on AMD GPUs,  
they are almost certainly using LoRA or QLoRA (LoRA + quantization, which is what we do here).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare quantized model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                         # rank -- higher = more capacity, more memory
    lora_alpha=32,                # scaling factor (alpha/r = effective learning rate scale)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable %:          {100 * trainable / total:.2f}%")
print(f"\nGPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 7. Train

**What happens during training:**
1. Batch of examples loaded to GPU memory
2. **Forward pass:** model predicts next token for each position
3. **Loss calculation:** cross-entropy between predicted and actual next tokens
4. **Backward pass:** compute gradients (how to adjust each LoRA weight)
5. **Optimizer step:** update LoRA weights using AdamW
6. Repeat

On T4 (16GB): ~15 examples/sec for this model size.  
On MI300X (192GB): could run much larger batch sizes, faster throughput.  
On MI355X (288GB): could train larger models without quantization at all.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=3,              # 3 passes over the data
    per_device_train_batch_size=2,   # limited by T4 memory
    gradient_accumulation_steps=4,   # effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_strategy="no",              # don't save checkpoints (Colab disk is small)
    fp16=True,                       # mixed precision -- uses fp16 for speed
    # bf16=True,                     # MI300X/MI355X supports bf16 natively (better for training)
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
)

print("Starting training...")
print(f"GPU memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

result = trainer.train()

print(f"\nTraining complete.")
print(f"Total training time: {result.metrics['train_runtime']:.1f} seconds")
print(f"Final loss: {result.metrics['train_loss']:.4f}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 8. Test Verilog generation AFTER fine-tuning

Compare outputs to the pre-training results above.  
With 200 real Verilog examples, the model should show noticeably better  
module structure, port declarations, and HDL style.

In [ ]:
print("=" * 60)
print("AFTER FINE-TUNING")
print("=" * 60)
for p in test_prompts:
    print(f"\nQ: {p}")
    print(f"A: {generate(p)}")
    print("-" * 40)

## 9. Inspect what LoRA actually changed

LoRA adapters are tiny. This is why fine-tuning is practical --  
you ship small adapter files, not entire model copies.

In [ ]:
# Save the LoRA adapter (not the full model)
model.save_pretrained("./lora_adapter")

import os
adapter_size = sum(
    os.path.getsize(os.path.join("./lora_adapter", f))
    for f in os.listdir("./lora_adapter")
    if os.path.isfile(os.path.join("./lora_adapter", f))
)
print(f"LoRA adapter size: {adapter_size / 1e6:.1f} MB")
print(f"Full model size:   ~2,200 MB (fp16)")
print(f"Adapter is {adapter_size / 2.2e9 * 100:.2f}% of the full model")
print(f"\nAdapter files:")
for f in os.listdir("./lora_adapter"):
    size = os.path.getsize(os.path.join("./lora_adapter", f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## 10. GPU memory breakdown

Understanding GPU memory is critical for the MI355X performance talks tomorrow.

In [ ]:
print("GPU Memory Breakdown")
print("=" * 50)
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
peak = torch.cuda.max_memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_mem / 1e9

print(f"Currently allocated: {allocated:.2f} GB")
print(f"Reserved by PyTorch: {reserved:.2f} GB")
print(f"Peak during training: {peak:.2f} GB")
print(f"Total GPU memory:    {total:.1f} GB")
print(f"Utilization:         {peak / total * 100:.1f}%")
print(f"")
print("What this means for AMD GPUs:")
print(f"  MI300X (192 GB): could fit {192 / peak:.0f}x this workload, or a ~{192 / peak * 1.1:.0f}B param model")
print(f"  MI355X (288 GB): could fit {288 / peak:.0f}x this workload, or a ~{288 / peak * 1.1:.0f}B param model")

---

## Summary: What You Just Did

1. **Loaded a model with 4-bit quantization** -- compressed 1.1B params to fit in 16GB
2. **Used real open-source Verilog data** -- 200 modules from `shailja/Verilog_GitHub` (109k total)
3. **Applied LoRA** -- made only ~1-3% of parameters trainable
4. **Ran supervised fine-tuning** -- forward pass, loss, backward pass, optimizer step
5. **Used mixed precision (fp16)** -- faster math, less memory
6. **Saved a tiny adapter** -- the LoRA weights are a few MB, not GB

### Open-source EDA/Verilog datasets available on HuggingFace:

| Dataset | Size | What it is |
|---------|------|-----------|
| `shailja/Verilog_GitHub` | 109k modules | Verilog code from GitHub (used here) |
| `JayZhang1/Verilogdata4pretrainCODET5` | 256k samples | Verilog for CodeT5 pretraining |
| `wangxinze/Verilog_data` | 187k samples | Verilog code collection |

### Questions this prepares you to ask tomorrow:

- **To AMD:** "Does bitsandbytes work on ROCm, or do I need a different quantization path for MI355X?"
- **To Eric Hartford:** "What LoRA rank and target modules do you use for fine-tuning 70B+ models on AMD?"
- **To TensorWave:** "On MI300X with 192GB HBM, could I train on the full 109k Verilog dataset without quantization?"
- **To Zyphra:** "How does your efficient architecture (Zamba) interact with LoRA? Do you still need adapters?"
- **To ZLUDA/Spectral Compute:** "Does the CUDA compatibility layer handle bitsandbytes and custom PEFT kernels?"